Starting with the 3D autoencoder

In [1]:
# using python 3.9 environment for cuvis, 3.8 for gdal 

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

import torch
from torchvision import datasets
from torchvision import transforms
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

!pip install tensorboard
!pip install tensorboardX
from tensorboardX import SummaryWriter

import os
import cv2

import glob 
# import cuvis

# !pip install GDAL
# !python3.9 -m pip install GDAL
from osgeo import gdal

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


     |████████████████████████████████| 5.5 MB 1.7 MB/s eta 0:00:01
     |████████████████████████████████| 226 kB 3.1 MB/s eta 0:00:01
     |████████████████████████████████| 189 kB 27.9 MB/s eta 0:00:01
     |████████████████████████████████| 5.6 MB 1.3 MB/s eta 0:00:011
     |████████████████████████████████| 105 kB 3.2 MB/s eta 0:00:01
     |████████████████████████████████| 133 kB 35.7 MB/s eta 0:00:01
     |████████████████████████████████| 181 kB 13.7 MB/s eta 0:00:01
     |████████████████████████████████| 151 kB 16.8 MB/s eta 0:00:01
     |████████████████████████████████| 85 kB 5.3 MB/s  eta 0:00:01
     |████████████████████████████████| 101 kB 7.5 MB/s eta 0:00:01


ModuleNotFoundError: No module named 'osgeo'

In [ ]:
def loadMeasurement(file_name):

    # load hyperspectral image
    _, extension = os.path.splitext(file_name)

    if extension == '.dat':
        # below is a way to load hyperspectral images that are tiff files
        mylist = []
        loaded, mylist = GDAL_imreadmulti(file_name)
        # loaded, mylist = cv2.imreadmulti(mats = mylist, filename = file_name, flags = cv2.IMREAD_ANYCOLOR )
        # print(mylist)
        cube=np.array(mylist)
        # print(cube.shape, loaded)
        cube = cube[:, :, :]
    else :
        print("Error: file type not supported")
        return

    # print(cube.shape)
    return cube

def GDAL_imreadmulti(file_name):
    # Open the dataset
    dataset = gdal.Open(file_name)

    # Check if opened
    if dataset:
      # print("Dataset opened...")
      width = dataset.RasterXSize
      height = dataset.RasterYSize
      num_bands = dataset.RasterCount

      image_bands = []

      for band_num in range(1, num_bands+1):
        band = dataset.GetRasterBand(band_num)

        # Read band data
        band_data = band.ReadAsArray()

        # Create an OpenCV Mat from the band data
        band_mat = np.array(band_data, dtype='float32')

        # Correct the orientation of the image
        band_mat = np.transpose(band_mat)
        band_mat = cv2.flip(band_mat, 1)

        # Apply threshold and convert to 8-bit unsigned integers
        _, band_mat = cv2.threshold(band_mat, 1.0, 1.0, cv2.THRESH_TRUNC)
        band_mat = cv2.convertScaleAbs(band_mat, alpha=(255.0))

        # Add the processed band to the list
        image_bands.append(band_mat)

      return True, image_bands

    else:
      print("GDAL Error: ", gdal.GetLastErrorMsg())
      return False, []
  
  
class AutoencoderDataset(Dataset):
    def __init__(self,  dataset_name, image_set,  root_dir, transform=None):
        self.dataset_name = dataset_name
        self.image_set = image_set
        self.transform = transform
        self.root = root_dir
        self.img_dir = root_dir
        # self.img_names = [f for f in os.listdir(self.img_dir) if f.endswith('.' + 'cu3')]
        self.img_names = [f for f in os.listdir(self.img_dir) if f.endswith('.' + 'dat')]
        print(self.img_names)

        self.num_images = len( self.img_names  )

    def __getitem__(self, index):
        
        hyperspectral_image1 =self.root + self.img_names[index]
        # print(hyperspectral_image1)
        
        #below is for cuvis 
        # if (use_load_measurement):
        #     hyperImage = loadMeasurement(userSettingsDir,hyperspectral_image1)
        # elif (use_reprocess_measurement):
        #     hyperImage = reprocessMeasurement(userSettingsDir,hyperspectral_image1,darkLoc,whiteLoc,distanceLoc,factoryDir)
        # data = hyperImage.astype(np.float32)
        
        hyperImage = loadMeasurement(hyperspectral_image1)
        data = hyperImage.astype(np.float32)/255
        
        # hyperImage = loadMeasurement(userSettingsDir,hyperspectral_image1)
        # hyperImage = reprocessMeasurement(userSettingsDir,hyperspectral_image1,darkLoc,whiteLoc,distanceLoc,factoryDir)
        
        # print(data)
        # print max value in data
        # print(np.max(data))
        
        # print(data.shape)
        # Assuming `data` is your numpy array with shape (51, 275, 290)
        data = np.expand_dims(data, axis=0)  # Add a new dimension, resulting in shape (1, 51, 275, 290)
        # print(data.shape)
        # Convert numpy array to PyTorch tensor
        data = torch.from_numpy(data)
        
        # Assuming `data` is your tensor with shape (51, 275, 290)
        # data = data.unsqueeze(0)  # Add batch dimension, resulting in shape (1, 51, 275, 290)
        # print(data.shape)
        # Resize data
        resized_data = F.interpolate(data, size=(273, 273), mode='bilinear', align_corners=False)

        resized_data = resized_data.squeeze(0)  # Remove batch dimension, resulting in shape (51, 275, 275)
        # print(resized_data.shape)
        # print(resized_data)
        return resized_data

    def __len__(self):
        return self.num_images
    
        
# Creating a PyTorch class
# 28*28 ==> 9 ==> 28*28
class spectral_AE_threeD(torch.nn.Module):
    def __init__(self):
        super().__init__()

        # Building an linear encoder with Linear
        # layer followed by Relu activation function
        # 140 ==> 3
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(num_data_points, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 8),
            torch.nn.ReLU(),
            torch.nn.Linear(8, 3),
        )

        # Building an linear decoder with Linear
        # layer followed by Relu activation function
        # The Sigmoid activation function
        # outputs the value between 0 and 1
        # 3 ==> 140
        self.decoder = torch.nn.Sequential(
            torch.nn.Linear(3, 8),
            torch.nn.ReLU(),
            torch.nn.Linear(8, 16),
            torch.nn.ReLU(),
            torch.nn.Linear(16, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, num_data_points),
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded    


class Autoencoder(torch.nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()
        
        # Encoder
        self.encoder =  torch.nn.Sequential(
            # torch.nn.Conv2d(275, 128, kernel_size=3, stride=2, padding=1),
            # torch.nn.ReLU(),
            # torch.nn.Conv2d(128, 64, kernel_size=3, stride=2, padding=1),
            # torch.nn.ReLU(),
            torch.nn.Conv2d(num_channels, 32, kernel_size=3, stride=2, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(32, 16, kernel_size=3, stride=2, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv2d(16, 3, kernel_size=3, stride=2, padding=1),
            # torch.nn.ReLU(),
        )
        
        # Decoder
        self.decoder =  torch.nn.Sequential(
            # torch.nn.ConvTranspose2d(3, 16, kernel_size=3, stride=2, padding=1, output_padding=0),
            # torch.nn.ReLU(),
            # torch.nn.ConvTranspose2d(16, 32, kernel_size=3, stride=2, padding=1, output_padding=0),
            # torch.nn.ReLU(),
            # torch.nn.ConvTranspose2d(32, 51, kernel_size=3, stride=2, padding=2, output_padding=1),
            # # torch.nn.ReLU(),
            # # torch.nn.ConvTranspose2d(64, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            # # torch.nn.ReLU(),
            # # torch.nn.ConvTranspose2d(128, 275, kernel_size=3, stride=2, padding=1, output_padding=1),
            # torch.nn.Sigmoid(),
            torch.nn.ConvTranspose2d(3, 16, kernel_size=3, stride=2, padding=1, output_padding=0),  # Adjust output_padding
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(16, 32, kernel_size=3, stride=2, padding=1, output_padding=0),  # Adjust output_padding
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(32, num_channels, kernel_size=3, stride=2, padding=1, output_padding=0),  # Adjust output_padding
            torch.nn.Sigmoid(),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return  encoded, decoded 
import torch
from torch import nn

class Autoencoder3(nn.Module):
    def __init__(self):
        super(Autoencoder3, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(num_channels, 100, kernel_size=10, stride=1, padding=(1,1)),
            nn.ReLU(),
            nn.Conv2d(100, 100, kernel_size=10, stride=1, padding=(1,1)),
            nn.ReLU(),
            nn.Conv2d(100, 50, kernel_size=5, stride=1, padding=(1,1)),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2, padding=(1,1)),
            nn.Conv2d(50, 50, kernel_size=5, stride=1, padding=(1,1)),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(50*5*5, 100),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(100, 1250),
            nn.ReLU(),
            nn.Unflatten(1, (50, 5, 5)),
            nn.Conv2d(50, 50, kernel_size=5, stride=1, padding=(1,1)),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='nearest'),
            nn.Conv2d(50, 50, kernel_size=5, stride=1, padding=(1,1)),
            nn.ReLU(),
            nn.Conv2d(50, 50, kernel_size=5, stride=1, padding=(1,1)),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x
    
class ConvAutoencoder3D(torch.nn.Module):  
    def __init__(self):
        super(ConvAutoencoder3D, self).__init__()
        
        # Encoder
        self.encoder =  torch.nn.Sequential(
            # torch.nn.Conv2d(275, 128, kernel_size=3, stride=2, padding=1),
            # torch.nn.ReLU(),
            # torch.nn.Conv2d(128, 64, kernel_size=3, stride=2, padding=1),
            # torch.nn.ReLU(),
            torch.nn.Conv3d(num_channels, 32, kernel_size=3, stride=2, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv3d(32, 16, kernel_size=3, stride=2, padding=1),
            torch.nn.ReLU(),
            torch.nn.Conv3d(16, 3, kernel_size=3, stride=2, padding=1),
            # torch.nn.ReLU(),
        )
        
        # Decoder
        self.decoder =  torch.nn.Sequential(
            # torch.nn.ConvTranspose2d(3, 16, kernel_size=3, stride=2, padding=1, output_padding=0),
            # torch.nn.ReLU(),
             # torch.nn.ConvTranspose2d(16, 32, kernel_size=3, stride=2, padding=1, output_padding=0),
            # torch.nn.ReLU(),
            # torch.nn.ConvTranspose2d(32, 51, kernel_size=3, stride=2, padding=2, output_padding=1),
            # # torch.nn.ReLU(),
            # # torch.nn.ConvTranspose2d(64, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            # # torch.nn.ReLU(),
            # # torch.nn.ConvTranspose2d(128, 275, kernel_size=3, stride=2, padding=1, output_padding=1),
            # torch.nn.Sigmoid(),
            torch.nn.ConvTranspose3d(3, 16, kernel_size=3, stride=2, padding=1, output_padding=0),  # Adjust output_padding
            torch.nn.ReLU(),
            torch.nn.ConvTranspose3d(16, 32, kernel_size=3, stride=2, padding=1, output_padding=0),  # Adjust output_padding
            torch.nn.ReLU(),
            torch.nn.ConvTranspose3d(32, num_channels, kernel_size=3, stride=2, padding=1, output_padding=0),  # Adjust output_padding
            torch.nn.Sigmoid(),
        )
          

    
def extract_rgb(cube, red_layer=78 , green_layer=40, blue_layer=25,  visualize=False):

  
    red_img = cube[ red_layer,:,:]
    green_img = cube[ green_layer,:,:]
    blue_img = cube[ blue_layer,:,:]

        
    data=np.stack([red_img,green_img,blue_img], axis=-1)
    # print(data.shape)
    #print(type(image))

    #convert to 8bit
    x_norm = (data-np.min(data))/(np.max(data)-np.min(data))
    image=(x_norm*255).astype('uint8')
    if visualize:
        #pass
        plt.imshow(image)
        plt.show()
    return image 

In [ ]:
# input_data_dir = "/workspaces/HyperImages/cornfields/session_002/" # need to reprocess this data
# userSettingsDir = "/workspaces/HyperTools/settings/ultris20/" 
# darkLoc = "../../HyperImages/cornfields/Calibration/dark__session_002_003_snapshot16423119279414228.cu3"
# whiteLoc = "../../HyperImages/cornfields/Calibration/white__session_002_752_snapshot16423136896447489.cu3"
# distanceLoc = "../../HyperImages/cornfields/Calibration/distanceCalib__session_000_790_snapshot16423004058237746.cu3"
# factoryDir = "../settings/ultris20/" # init.daq file
# outDir ="../../HyperImages/export/"
# dataset='cornfields'
# use_reprocess_measurement=True
# use_load_measurement=False
# num_channels = 164


# input_data_dir = "/workspaces/HyperImages/cornflakes_and_pasta/cornflakes_and_noodles_000/"
# userSettingsDir = "/workspaces/HyperTools/settings/ultris20/" 
# dataset='cornflakes_and_pasta'
# use_load_measurement=True
# num_channels = 51


input_data_dir = "/workspaces/HyperImages/LIB-HSI/LIB-HSI/test/reflectance_cubes/"
dataset='lib_hsi'
num_channels = 204


full_dataset = AutoencoderDataset(dataset_name=dataset, root_dir=input_data_dir, image_set='train')

spectral_train_data, spectral_test_data = train_test_split(
    full_dataset,  test_size=0.3, random_state=21
)
train_loader = DataLoader(spectral_train_data, batch_size=8, shuffle=True,num_workers=8)
test_loader = DataLoader(spectral_test_data, batch_size=8,num_workers=8)

print(len(spectral_test_data), len(spectral_train_data), full_dataset.num_images)

In [ ]:
example= full_dataset[3]

print(example.shape)

if num_channels == 51:
    example1_image = extract_rgb(example, 31,13,2)
else:
    example1_image = extract_rgb(example)

plt.imshow(example1_image)
plt.show()

num_data_points = example.shape[0] * example.shape[1] * example.shape[2]

In [ ]:

# !pip install tensorboard
from tensorboardX import SummaryWriter# create a summary writer (to track progress of the model )

writer = SummaryWriter()

# add port and click load \type url into web browser

# click globe in vscode on correct port
%load_ext tensorboard
%tensorboard --logdir runs --port 6006

In [ ]:




#for 2D model
#spectral_model = ConvAutoencoder()

#for 3D model
#spectral_model = ConvAutoencoder3D()

spectral_model = Autoencoder()


loss_function = torch.nn.L1Loss()
criterion = torch.nn.MSELoss()
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print (device)
spectral_model = spectral_model.to(device)
optimizer = torch.optim.Adam(spectral_model.parameters(),
                             lr = 1e-3,
                             weight_decay = 1e-4)

epochs = 50
history_loss_train = []
history_loss_val = []
for epoch in range(epochs):
    spectral_model.train()
    for inputs in train_loader:
        
        
       #added these lines to squeeze a 1 into index 2 for 3d model
        # print(inputs.shape)
        # inputs = inputs.unsqueeze(2)
        # print(inputs.shape)
        ############################

        inputs = inputs.float().to(device)
        encoded, outputs = spectral_model(inputs)
        # print(inputs.shape, outputs.shape)
        loss = criterion(outputs, inputs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    spectral_model.eval()
    with torch.no_grad():
        for inputs in test_loader:
            # inputs = inputs.view(inputs.size(0), -1)  # Flatten the inputs

            inputs = inputs.float().to(device)
            
            #added these lines to squeeze a 1 into index 2
            # print(inputs.shape)
            # inputs = inputs.unsqueeze(2)
            # print(inputs.shape)
            ############################
            
            encoded, outputs = spectral_model(inputs)
            val_loss = criterion(outputs, inputs)
    history_loss_train.append(loss.item())
    history_loss_val.append(val_loss.item())
    print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}, Test Loss: {val_loss.item()}')
    writer.add_scalar('data/train_loss', loss.item(), epoch)
    writer.add_scalar('data/test_loss', val_loss.item(), epoch)
    
    


In [ ]:
plt.plot(history_loss_train, label="Training Loss")
plt.plot(history_loss_val, label="Validation Loss")
plt.legend()

writer.close()

In [ ]:
import matplotlib.pyplot as plt





# # Get a batch of test data
# test_data_iter = iter(test_loader)
# inputs = next(test_data_iter)
# Get the first batch of test data
first_batch = next(iter(test_loader))

# Select the first item from the batch
first_item = first_batch[0]
# Move the inputs to the device
first_item = first_item.float().to(device)

# Get the model's reconstruction of the inputs
encoded, outputs = spectral_model(first_item)

# Move the inputs and outputs to the CPU for plotting
first_item = first_item.cpu()
outputs = outputs.cpu()

# Detach the inputs and outputs from the computation graph
first_item = first_item.detach().numpy()
outputs = outputs.detach().numpy()

print(first_item.shape, outputs.shape, encoded.shape)

# Transpose the dimensions of the encoded representation
encoded_transposed = np.transpose(encoded.cpu().detach().numpy(), (1, 2, 0))
print(encoded_transposed.shape)




if num_channels == 51:
    example_image = extract_rgb(first_item, 31,13,2)
    example_image2 = extract_rgb(outputs, 31,13,2)
else:
    example_image = extract_rgb(first_item)
    example_image2 = extract_rgb(outputs)
    
    

# Initialize an empty array for the normalized data
normalized = np.empty_like(encoded_transposed)

# Iterate over each channel
for i in range(encoded_transposed.shape[2]):
    # Get the current channel
    channel = encoded_transposed[:, :, i]

    # Normalize the current channel
    normalized_channel = (channel - channel.min()) / (channel.max() - channel.min())

    # Store the normalized channel
    normalized[:, :, i] = normalized_channel
    
    
    

fig = plt.figure(figsize=(20, 40))

plt.subplot(5, 1, 1)  # 1 row, 3 columns, 1st subplot = image1
plt.imshow(example_image)
plt.title('Input Hyperspectral Image RGB')

plt.subplot(5, 1, 2)  # 1 row, 3 columns, 1st subplot = image1
plt.imshow(example_image2)
plt.title('Output Reconstructed HyperSpectral Image RGB')


plt.subplot(5, 1, 3)  # 1 row, 3 columns, 1st subplot = image1
plt.imshow(normalized)
plt.title('Encoded Hyperspectral Image')


plt.show()


Now moving on to the segmentation modules